# Advanced Programme in Deep Learning (Foundations and Applications)
## A Program by IISc and TalentSprint

### Mini Project Notebook: To perform text classification of coronavirus tweets during the peak Covid - 19 period using LSTMs/RNNs/CNNs/BERT.


## Learning Objectives

At the end of the mini-hackathon, you will be able to :

* perform data preprocessing/preprocess the text
* represent the text/words using the pretrained word embeddings - Word2Vec/Glove
* build the deep neural network (RNN, LSTM, GRU, CNNs, Bidirectional-LSTM, GRU, BERT) to classify the tweets


### Introduction

First we need to understand why sentiment analysis is needed for social media?

People from all around the world have been using social media more than ever. Sentiment analysis on social media data helps to understand the wider public opinion about certain topics such as movies, events, politics, sports, and more and gain valuable insights from this social data. Sentiment analysis has some powerful applications. Nowadays it is also used by some businesses to do market research and understand the customer’s experiences for their products or services.

Now an interesting question about this type of problem statement that may arise in your mind is that why sentiment analysis on COVID-19 Tweets? What is about the coronavirus tweets that would be positive? You may have heard sentiment analysis on movie or book reviews, but what is the purpose of exploring and analyzing this type of data?

The use of social media for communication during the time of crisis has increased remarkably over the recent years. As mentioned above, analyzing social media data is important as it helps understand public sentiment. During the coronavirus pandemic, many people took to social media to express their anger, grief, or sadness while some also spread happiness and positivity. People also used social media to ask their network for help related to vaccines or hospitals during this hard time. Many issues related to this pandemic can also be solved if experts considered this social data. That’s the reason why analyzing this type of data is important to understand the overall issues faced by people.



## Dataset

The given challenge is to build a multiclass classification model to predict the sentiment of Covid-19 tweets. The tweets have been pulled from Twitter and manual tagging has been done. We are given information like Location, Tweet At, Original Tweet, and Sentiment.

The training dataset consists of 36000 tweets and the testing dataset consists of 8955 tweets. There are 5 sentiments namely ‘Positive’, ‘Extremely Positive’, ‘Negative’, ‘Extremely Negative’, and ‘Neutral’ in the sentiment column.

## Description

This dataset has the following information about the user who tweeted:

1. **UserName:** twitter handler
2. **ScreenName:** a personal identifier on Twitter and is separate from the username
3. **Location:** where in the world the person tweets from
4. **TweetAt:** date of the tweet posted (DD-MM-YYYY)
5. **OriginalTweet:** the tweet itself
6. **Sentiment:** sentiment value



## Problem Statement

To build and implement a multiclass classification deep neural network model to classify between Positive/Extremely Positive/Negative/Extremely Negative/Neutral sentiments

## Grading = 10 Marks

Here is a handy link to Kaggle's competition documentation (https://www.kaggle.com/docs/competitions), which includes, among other things, instructions on submitting predictions (https://www.kaggle.com/docs/competitions#making-a-submission).

## Instructions for downloading train and test dataset from Kaggle API are as follows:

### 1. Create an API key in Kaggle.

To do this, go to the competition site on Kaggle at (https://www.kaggle.com/t/15cef0def403469ebbb5db1a67991873) and open your user settings page. Click Account.

* Click on your profile picture at the top-right corner of the page.

![alt text](https://i.imgur.com/kSLmEj2.png)

* In the popout menu, click the Settings option.

![alt text](https://i.imgur.com/tNi6yun.png)








### 2. Next, scroll down to the API access section and click generate to download an API key (kaggle.json).
![alt text](https://i.imgur.com/vRNBgrF.png)


### 3. Upload your kaggle.json file using the following snippet in a code cell:



In [ ]:
# from google.colab import files
# files.upload()

Check if files are uploaded

In [ ]:
# #If successfully uploaded in the above step, the 'ls' command here should display the kaggle.json file.
# %ls

### 4. Install the Kaggle API using the following command


In [ ]:
# !pip install -U -q kaggle==1.5.8

### 5. Move the kaggle.json file into ~/.kaggle, which is where the API client expects your token to be located:



In [ ]:
# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/kaggle.json

In [ ]:
# Execute the following command to verify whether the kaggle.json is stored in the appropriate location: ~/.kaggle/kaggle.json
# !ls ~/.kaggle

In [ ]:
# !chmod 600 /root/.kaggle/kaggle.json # run this command to ensure your Kaggle API token is secure on colab

Setup glove

In [ ]:
#setup
from IPython import get_ipython
import warnings
warnings.filterwarnings("ignore")

ipython = get_ipython()
ipython.magic("sx wget -qq https://cdn.iiith.talentsprint.com/aiml/Experiment_related_data/glove.6B.zip")
ipython.magic("sx unzip glove.6B.zip")

### 6. Now download the Test Data from Kaggle

**NOTE: If you get a '404 - Not Found' error after running the cell below, it is most likely that the user (whose kaggle.json is uploaded above) has not 'accepted' the rules of the competition and therefore has 'not joined' the competition.**

If you encounter **401-unauthorised** download latest **kaggle.json** by repeating steps 1 & 2

no need of this if you do below 2 steps of downloading from kaggle and upload it  is done

In [ ]:
# #If you get a forbidden link, you have most likely not joined the competition.
# !kaggle competitions download -c multi-text-classification-of-coronavirus-tweets

upload data zip file from kaggle to-classify-coronavirus-tweets-during-covid-19.zip https://www.kaggle.com/competitions/to-classify-coronavirus-tweets-during-covid-19/data

In [ ]:
 from google.colab import files
 files.upload()

In [ ]:
!unzip /content/to-classify-coronavirus-tweets-during-covid-19.zip

## YOUR CODING STARTS FROM HERE

## Import required packages

In [ ]:
# Import required packages
import pandas as pd
import numpy as np
from sklearn.utils import shuffle

import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
from nltk.corpus import stopwords
import string
from gensim.utils import simple_preprocess  # utility function to preprocess text (tokenization, lowering, removing punctuation)

import tensorflow as tf  # use TensorFlow
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences  # Padding sequences for uniform input length
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.layers import Input, Embedding, Dense, Bidirectional, Dropout, GRU   # Key layers used in NLP models
from tensorflow.keras.models import Sequential    # the model

##   **Stage 1**:  Data Loading and Perform Exploratory Data Analysis (1 Points)

* Load the Dataset


In [ ]:
# Read the CSV file directly using pandas
df = pd.read_csv('/content/corona_nlp_train.csv/corona_nlp_train.csv', encoding='ISO-8859-1')

# Extract the desired columns: Location (column 2), OriginalTweet (column 4), Sentiment (column 5)
selected_columns = df[['Location', 'OriginalTweet', 'Sentiment']]

# Display the DataFrame
selected_columns.head()

* Check for Missing Values

In [ ]:
# Check for missing values in the DataFrame
missing_values = df.isnull().sum()

# Display the count of missing values for each column
print(missing_values)

* Visualize the sentiment column values


In [ ]:
import seaborn as sns

import matplotlib.pyplot as plt

# Count the occurrences of each sentiment
sentiment_counts = df['Sentiment'].value_counts()

# Create a bar plot
plt.figure(figsize=(10, 6))
sns.barplot(x=sentiment_counts.index, y=sentiment_counts.values, palette='viridis')
plt.title('Distribution of Sentiment Values')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.xticks(rotation=45)  # Rotate x labels for better readability
plt.show()

* Visualize top 10 Countries that had the highest tweets using countplot (Tweet count vs Location)


In [ ]:


# Count the number of tweets per location
location_counts = df['Location'].value_counts().head(10)  # Get top 10 locations

# Create a DataFrame for visualization
top_locations_df = location_counts.reset_index()
top_locations_df.columns = ['Location', 'TweetCount']

# Set the aesthetics for the plot
plt.figure(figsize=(10, 6))
sns.set_theme(style="whitegrid")

# Create a count plot
sns.barplot(x='TweetCount', y='Location', data=top_locations_df, palette='viridis')

# Add labels and title
plt.xlabel('Number of Tweets')
plt.ylabel('Location')
plt.title('Top 10 Countries with Highest Tweets')
plt.show()

* Plotting Pie Chart for the Sentiments in percentage


In [ ]:
# Count the occurrences of each sentiment
sentiment_counts = df['Sentiment'].value_counts()

# Prepare data for pie chart
labels = sentiment_counts.index
sizes = sentiment_counts.values
colors = ['gold', 'lightcoral', 'lightskyblue']  # Customize colors based on your sentiments
explode = [0.1] * len(labels)  # explode all slices for better visibility

# Create a pie chart
plt.figure(figsize=(8, 6))
plt.pie(sizes, explode=explode, labels=labels, colors=colors,
        autopct='%1.1f%%', shadow=True, startangle=140)

# Equal aspect ratio ensures the pie chart is circular
plt.axis('equal')
plt.title('Distribution of Sentiments')
plt.show()

* WordCloud for the Tweets/Text

    * Visualize the most commonly used words in each sentiment using wordcloud
    * Refer to the following [link](https://medium.com/analytics-vidhya/word-cloud-a-text-visualization-tool-fb7348fbf502) for Word Cloud: A Text Visualization tool




In [ ]:
from wordcloud import WordCloud
import time  # Import the time module
import re  # Regular expression module


# Create a function to generate word clouds for each sentiment
def generate_wordcloud(data, sentiment):
    text = ' '.join(data)
    wordcloud = WordCloud(width=800, height=400, background_color='white',
                          max_words=200).generate(text)

    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')  # Turn off the axis
    plt.title(f'Word Cloud for {sentiment} Sentiment')
    plt.show()



# Generate word clouds for each sentiment with a delay
sentiments = df['Sentiment'].unique()
for sentiment in sentiments:
    sentiment_data = df[df['Sentiment'] == sentiment]['OriginalTweet']
    generate_wordcloud(sentiment_data, sentiment)

    # # Add a delay of 3 seconds before the next plot
    # time.sleep(3)

##   **Stage 2**: Data Pre-Processing  (2 Points)

####  Clean and Transform the data into a specified format


In [ ]:
nltk.download('punkt_tab')

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from scipy import stats
from sklearn.feature_extraction.text import TfidfVectorizer


# nltk.download('punkt')
# nltk.download('wordnet')
# nltk.download('stopwords')

# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()

#function to fill blank  in location as unknown

df['Location'].fillna('Unknown', inplace=True)
# Display the count of missing values for each column
print(df.isnull().sum())


# Function to clean text data
def clean_text(text):
    # Convert to lowercase
    text = text.lower()
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '[URL]', text, flags=re.MULTILINE)
    # Remove punctuation, numbers, and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Tokenization
    words = word_tokenize(text)
    # Remove stop words
    stop_words = set(stopwords.words('english'))
    words = [word for word in words if word not in stop_words]
    # Lemmatization
    words = [lemmatizer.lemmatize(word) for word in words]
    # Join words back into a single string
    return ' '.join(words)

from sklearn.ensemble import IsolationForest


# Apply the cleaning function to the 'OriginalTweet' column
df['CleanedTweet'] = df['OriginalTweet'].apply(clean_text)

# Display the cleaned tweets
print(df[['OriginalTweet', 'CleanedTweet']].head())

# Step 2: Define a method for outlier detection (e.g., based on length of cleaned tweets)
df['TweetLength'] = df['CleanedTweet'].apply(len)
# Calculate z-scores for tweet lengths to identify outliers
from scipy import stats

z_scores = np.abs(stats.zscore(df['TweetLength']))
df['Z-Score'] = z_scores

# Identify outliers
threshold = 2  # Common threshold
outliers = df[df['Z-Score'] > threshold]  # Common threshold for outlier detection
print("Outliers based on Tweet Length:")
print(outliers)

# Step 3: Remove outliers from the DataFrame
df = df[df['Z-Score'] <= threshold]

# Step 4: Split the data into training and test sets
from sklearn.model_selection import train_test_split



In [ ]:
# Generate word clouds for each sentiment on cleaned data
sentiments = df['Sentiment'].unique()
for sentiment in sentiments:
    sentiment_data = df[df['Sentiment'] == sentiment]['CleanedTweet']
    generate_wordcloud(sentiment_data, sentiment)

In [ ]:
# Calculate the length of each tweet
df['TweetLength'] = df['OriginalTweet'].apply(len)

# Plotting
plt.figure(figsize=(12, 6))

# Histogram
plt.subplot(1, 2, 1)
sns.histplot(df['TweetLength'], bins=30, kde=True)
plt.title('Distribution of Tweet Lengths')
plt.xlabel('Tweet Length')
plt.ylabel('Frequency')

# Boxplot
plt.subplot(1, 2, 2)
sns.boxplot(x=df['TweetLength'])
plt.title('Boxplot of Tweet Lengths')
plt.xlabel('Tweet Length')

plt.tight_layout()
plt.show()

In [ ]:

# Calculate statistics
median_length = df['TweetLength'].median()
percentile_75 = df['TweetLength'].quantile(0.75)
max_length = df['TweetLength'].max()

# Print the statistics
print(f"Median Tweet Length: {median_length}")
print(f"75th Percentile Tweet Length: {percentile_75}")
print(f"Maximum Tweet Length: {max_length}")
# Many social media posts, including tweets, often fall within the range of 50 to 150 characters. Setting MAX_SENT_LEN to 150
recommended_max_sent_len = min(150, max_length)  # Limit to maximum length as well
print(f"Recommended Maximum Sentence Length: {recommended_max_sent_len}")


##   **Stage 3**: Build the Word Embeddings using pretrained Word2vec/Glove (Text Representation) (1 Point)



In [ ]:
# Step 1: Initialize the Tokenizer
tokenizer = Tokenizer()

# Step 2: Fit the Tokenizer on the CleanedTweet column
tokenizer.fit_on_texts(df['CleanedTweet'])

# Step 3: Convert all texts to sequences of integers
sequences = tokenizer.texts_to_sequences(df['CleanedTweet'])



# Step 2: Convert texts to sequences
# Parameters
embedding_dim = 200  # Embedding dimension of pre-trained embeddings (e.g., 300 for GloVe 300d)
vocab_size = len(tokenizer.word_index) + 1  # Vocabulary size including padding
MAX_SENT_LEN = 30 #recommended_max_sent_len  # Adjust based on your text length

# Example: Loading pre-trained GloVe embeddings (if not done already)
embeddings_index = {}
with open('glove.6B.200d.txt', 'r', encoding='utf-8') as f:  # Update path as necessary
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs

In [ ]:
embedding_matrix = np.zeros((vocab_size, embedding_dim))
words_not_found = []
for word, i in tokenizer.word_index.items():
    if i >= vocab_size:
        continue
    embedding_vector = embeddings_index.get(word)
    # embedding_vector = embeddings_index[word]
    if (embedding_vector is not None) and len(embedding_vector) > 0:
                embedding_matrix[i] = embedding_vector
    else:
        words_not_found.append(word)

In [ ]:
X = pad_sequences(sequences, maxlen=MAX_SENT_LEN)

labels = df['Sentiment'].values
# Encode labels
le = LabelEncoder()
Y = le.fit_transform(labels)  # Convert categorical labels to integers
# Split the dataset into training and testing sets
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [ ]:
print(len(tokenizer.word_index))

In [ ]:
print(tokenizer.word_index)


In [ ]:
print("Unique values in y_train:", np.unique(y_train))
print("Unique values in y_test:", np.unique(y_test))

In [ ]:
max_length_tweet = max(df['OriginalTweet'], key=len)
print("Max length tweet:", max_length_tweet)
print("Length of max length tweet:", len(max_length_tweet))

min_length_tweet = min(df['OriginalTweet'], key=len)  # Replace 'column_name' with the actual column name
print("Min length tweet:", min_length_tweet)
print("Length of min length tweet:", len(min_length_tweet))

##   **Stage 4**: Build and Train the Deep Recurrent Model using Pytorch/Keras (4 Points)



Preprocess the text data, including tokenization and padding

GRU Model using keras


In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, GRU, Dense, Dropout, Bidirectional, Layer
import tensorflow as tf

# Custom Attention Layer
class Attention(Layer):
    def __init__(self, **kwargs):
        super(Attention, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(name="attention_weight", shape=(input_shape[-1], 1), initializer="random_normal")
        self.b = self.add_weight(name="attention_bias", shape=(input_shape[1], 1), initializer="zeros")
        super(Attention, self).build(input_shape)

    def call(self, inputs):
        score = tf.nn.tanh(tf.matmul(inputs, self.W) + self.b)
        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = attention_weights * inputs
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector


In [ ]:
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.layers import Conv1D, MaxPooling1D,GlobalMaxPooling1D
from tensorflow.keras.initializers import HeNormal

# Model Architecture
embedding_dim = 200
vocab_size = len(tokenizer.word_index) + 1
MAX_SENT_LEN = 30

# Input layer
input = Input(shape=(MAX_SENT_LEN,))

# Embedding layer
embedding = Embedding(input_dim=vocab_size, output_dim=embedding_dim, weights=[embedding_matrix],
                      input_length=MAX_SENT_LEN, trainable=False)(input)
# CNN layer
cnn_layer = Conv1D(filters=128, kernel_size=3, activation='relu',padding='same')(embedding)
cnn_layer = MaxPooling1D(pool_size=2)(cnn_layer)
cnn_pool = GlobalMaxPooling1D()(cnn_layer)

# LSTM layer
lstm_out = Bidirectional(LSTM(64, return_sequences=True))(embedding)

# GRU layer
gru_out = Bidirectional(GRU(64, return_sequences=True))(lstm_out)

# Attention layer
attention = Attention()(gru_out)


# Combine CNN and GRU outputs
combined = concatenate([cnn_pool, attention])  # Output shape: (None, 256)


# Fully connected layers
dense1 = Dense(64, activation="relu",kernel_initializer=HeNormal())(combined)
dropout = Dropout(0.5)(dense1)
output = Dense(5, activation="softmax")(dropout)  # Use sigmoid for binary classification; softmax for multi-class

# Define model
model = Model(inputs=input, outputs=output)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])  # Change to categorical_crossentropy for multi-class

# Summary
model.summary()



In [ ]:
print(cnn_layer.shape)
print(lstm_out.shape)

In [ ]:
print(np.isnan(embedding_matrix).any(), np.isinf(embedding_matrix).any())
print(np.min(embedding_matrix), np.max(embedding_matrix))

Train model

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# model.compile(optimizer=Adam(learning_rate=0.0001), loss='binary_crossentropy', metrics=['accuracy'])
# model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
# Compile the model
# model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])


# Now you can train the model
# Assuming you have your training data ready as X_train_padded and y_train
BATCH_SIZE = 32
N_EPOCHS = 1
history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=N_EPOCHS, batch_size=BATCH_SIZE)

In [ ]:
# Print the entire history
print(history.history)

In [ ]:
mdl='my_attn_model.keras'

In [ ]:
model.save(mdl)


In [ ]:

# # Later, you can load the model and continue training
# from keras.models import load_model

# model = load_model(mdl,custom_objects={'Attention': attention)
# history2 = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=1, batch_size=BATCH_SIZE)
#  model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test), initial_epoch=15)

##   **Stage 5**: Evaluate the Model and get model predictions on the test dataset (2 Points)

* Upload the model predictions to kaggle by mapping the sentiment column vlalues from numericals the categorical







In [ ]:
# Evaluate model on test data
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=1)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
# print('Testing...')
# model.evaluate(X_test, y_test)

 Get Predictions on Test Data

In [ ]:
# Load the test data
test_df = pd.read_csv('/content/corona_nlp_test.csv/corona_nlp_test.csv',encoding='ISO-8859-1')

# Preprocess the test data (using the same tokenizer)
test_sequences_pred = tokenizer.texts_to_sequences(test_df['OriginalTweet'])
# MAX_SENT_LEN = 30  # This should match your training max length
X_test_pred = pad_sequences(test_sequences_pred, maxlen=MAX_SENT_LEN)



In [ ]:
# Get predictions
predictions = model.predict(X_test_pred)

# Get the class with the maximum probability
predicted_classes = np.argmax(predictions, axis=1)

# Assuming you have a mapping of classes back to their original labels
labels_mapping = le.inverse_transform(predicted_classes)  # Decode class integers to labels

In [ ]:
# Prepare the submission DataFrame
submission_df = pd.DataFrame({
    'Test_Id': test_df['UserName'],  # Assuming 'UserName' serves as the ID
    'Sentiment': labels_mapping
})

# Save the DataFrame to a CSV file
submission_df.to_csv('submission.csv', index=False)  # Save without the index

### Instructions for preparing Kaggle competition predictions


* Get the predictions using trained model and prepare a csv file
    * DeepNet model gives output for each class, consider the maximum value among all classes as prediction using `np.argmax`.

* Predictions (csv) file should contain 2 columns as Sample_Submission.csv
  - First column is the Test_Id which is considered as index
  - Second column is prediction in decoded form (for eg. Positive, Negative etc...).

BERT

In [ ]:
!pip show tensorflow transformers

In [ ]:
bert_model = TFBertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=5)
bert_model = bert_model.cuda()

# Compile the model using TensorFlow's Keras API
bert_model.compile(
    optimizer='adam',  # Correct reference to Adam optimizer
    loss='sparse_categorical_crossentropy',  # For sparse labels
    metrics=['accuracy']
)